# 📚 Notebook 1 — Build the Basic RAG Pipeline
## RAGAS Evaluation Series · Part 1 of 4

---

## 🎯 What this notebook does

Before we can evaluate a RAG system, we need one to evaluate.  
This notebook builds the **exact same Basic RAG pipeline** from the previous series — but with one important addition: we **save every question, answer, and retrieved context** to a file so the evaluation notebooks can use them.

```
Notebook 1  →  Build RAG + save outputs
Notebook 2  →  Create evaluation dataset (questions + ground truths)
Notebook 3  →  Run RAG, capture Question / Answer / Contexts
Notebook 4  →  Run RAGAS, get Faithfulness / Relevancy / Precision / Recall
```

> 💡 **Why save outputs?**  
> RAGAS needs: the question, the answer, the retrieved chunks, and optionally a ground truth.  
> We capture all of these during the RAG run so Notebook 4 can evaluate them without re-running the pipeline.

## 📦 Prerequisites & Installation

```bash
pip install langchain langchain-community langchain-ollama langchain-chroma chromadb
```

**Ollama running with models pulled:**
```bash
ollama pull nomic-embed-text
ollama pull gpt-oss:120b-cloud
```

**Your document at** `docs/company_policy.txt`

In [1]:
# ── Install (uncomment if needed) ────────────────────────────────────────────
# !pip install langchain langchain-community langchain-ollama langchain-chroma chromadb

---
## 📚 Step 0 — Imports

In [2]:
from langchain_community.document_loaders.text import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
import json

print("✅ All imports successful")

C:\Users\sr43993\AppData\Local\Temp\ipykernel_15112\223754098.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader
d:\RAG Practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports successful


---
## 📄 Steps 1–5 — Build the RAG Pipeline

Same pipeline as the Basic RAG notebook.  
Refer to that notebook for deep-dive explanations of each step.

In [3]:
# ── Step 1: Load Document ────────────────────────────────────────────────────
loader = TextLoader("docs/company_policy.txt", encoding="utf-8")
documents = loader.load()
print(f"✅ Loaded {len(documents)} document(s)")
print(f"\n📄 Preview: {documents[0].page_content[:300]}")

✅ Loaded 1 document(s)

📄 Preview: Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.


In [4]:
# ── Step 2: Split into Chunks ────────────────────────────────────────────────
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)
print(f"✅ Total Chunks: {len(chunks)}")

✅ Total Chunks: 1


In [5]:
# ── Step 3: Embeddings + Vector Store ────────────────────────────────────────
print("Loading embedding model...\n")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"✅ Vector store ready — {vectorstore._collection.count()} vectors stored")

Loading embedding model...

✅ Vector store ready — 1 vectors stored


In [6]:
# ── Step 4: Load LLM ─────────────────────────────────────────────────────────
llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)
print("✅ LLM ready")

✅ LLM ready


---
## 💾 Step 5 — RAG Function with Output Capture

This is the key addition for RAGAS evaluation.  
Every time we call `run_rag()`, it saves:
- The **question** asked
- The **answer** generated
- The **retrieved contexts** (list of chunk texts)

These three are exactly what RAGAS needs to compute its metrics.

In [7]:
# ── Step 5: RAG function that captures outputs for evaluation ─────────────────
#
# We return a dict with question, answer, and contexts because RAGAS needs all three.
# This is the standard RAGAS input format.

def run_rag(question: str) -> dict:
    """
    Run the full RAG pipeline and return a dict with:
      - question:  the original question
      - answer:    the LLM's generated answer
      - contexts:  list of retrieved chunk texts (not Document objects)
    """
    # Retrieve top-3 chunks
    retrieved_docs = retriever.invoke(question)
    contexts = [doc.page_content for doc in retrieved_docs]

    # Build context string
    context = "\n\n".join(contexts)

    # Craft prompt
    prompt = f"""You are a helpful assistant.
Answer ONLY using the provided context.
If the answer is not present, reply: "I could not find that information in the document."

Context:
{context}

Question:
{question}
"""
    # Generate answer
    response = llm.invoke(prompt)

    return {
        "question":  question,
        "answer":    response.content,
        "contexts":  contexts,   # list of strings — RAGAS expects this format
    }

print("✅ run_rag() function defined")

✅ run_rag() function defined


---
## 🧪 Step 6 — Test the Pipeline

Let's run a quick test to confirm everything works before saving outputs.

In [8]:
# ── Quick test ────────────────────────────────────────────────────────────────
test = run_rag("What is the maternity leave policy?")

print(f"❓ Question : {test['question']}")
print(f"\n🤖 Answer   : {test['answer']}")
print(f"\n📄 Contexts ({len(test['contexts'])} chunks retrieved):")
for i, ctx in enumerate(test['contexts'], 1):
    print(f"  [{i}] {ctx[:120]}...")

❓ Question : What is the maternity leave policy?

🤖 Answer   : Employees are eligible for 6 months of maternity leave.

📄 Contexts (1 chunks retrieved):
  [1] Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

An...


---
## 💾 Step 7 — Save RAG Outputs to File

We save the pipeline outputs so Notebooks 3 and 4 can load them without re-running everything.  
The file format is **JSON Lines** (one dict per line) — easy to append to incrementally.

In [9]:
# ── Save the RAG function and vector store for use in Notebook 3 ─────────────
# We save these as module-level globals so Notebook 3 can import or re-create them.
# For the demo, we also run a batch of test questions and save the results.

# Test questions — replace or extend with your own
test_questions = [
    "What is the maternity leave policy?",
    "How many days annual leave do employees get?",
    "What is the notice period before resignation?",
    "Can employees work from home?",
    "What is the probation period for new employees?",
]

rag_outputs = []
for q in test_questions:
    result = run_rag(q)
    rag_outputs.append(result)
    print(f"✅ {q}")
    print(f"   → {result['answer'][:80]}...")
    print()

# Save to file for Notebook 3 to load
with open("rag_outputs.json", "w") as f:
    json.dump(rag_outputs, f, indent=2)

print(f"\n💾 Saved {len(rag_outputs)} RAG outputs to rag_outputs.json")
print("\n➡️  Next: open Notebook 2 to create the evaluation dataset.")

✅ What is the maternity leave policy?
   → Employees are eligible for 6 months maternity leave....

✅ How many days annual leave do employees get?
   → Employees receive 24 days of annual leave....

✅ What is the notice period before resignation?
   → The notice period before resignation is 60 days....

✅ Can employees work from home?
   → Yes. Employees are permitted to work from home twice a week....

✅ What is the probation period for new employees?
   → I could not find that information in the document....


💾 Saved 5 RAG outputs to rag_outputs.json

➡️  Next: open Notebook 2 to create the evaluation dataset.
